In [43]:
!pip install -r requirements.txt

2333.32s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [44]:
import torch
import pandas as pd
import whisper
import torchcodec
import soundfile as sf
import jiwer
import json

In [45]:
# from datasets import load_dataset

# # This will now use your 'hf auth' credentials to bypass the gate
# giga = load_dataset(
#     "speechcolab/gigaspeech", 
#     "xs", 
#     split="test", 
#     token=True  # This tells the library to look for the token you just saved
# )

# print("Dataset loaded and cached successfully!")

In [46]:
from datasets import load_dataset

test_data = load_dataset(
    "speechcolab/gigaspeech",
    "xs",
    split="test",
    cache_dir="/home/cynos1/Downloads/Audio2txtprism/hf_datasets"
)

In [47]:
df= test_data.to_pandas()
print(df.isnull().sum())

segment_id            0
speaker               0
text                  0
audio                 0
begin_time            0
end_time              0
audio_id              0
title                 0
url                   0
source                0
category              0
original_full_path    0
dtype: int64


In [48]:
empty_texts = df["text"].isna().sum() + (df["text"].str.strip() == "").sum()

print("Empty texts:", empty_texts)

Empty texts: 0


In [49]:
!pip install torchcodec

2342.13s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [50]:
import sys
print(sys.executable)

/home/cynos1/Downloads/Audio2txtprism/venv/bin/python


In [51]:
print(len(test_data))
print(test_data[0])

25619
{'segment_id': 'YOU1000000134_S0000042', 'speaker': 'N/A', 'text': 'ONE OF THEIR STANFORD PROFESSORS USED TO SAY <COMMA> WELL <COMMA> THE DIFFERENCE BETWEEN THE TWO OF THEM WAS THAT SERGEI WOULD JUST BURST INTO MY OFFICE WITHOUT ASKING <PERIOD> LARRY WOULD KNOCK AND THEN BURST IN <PERIOD>', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0xed4efc2a10d0>, 'begin_time': 223.66200256347656, 'end_time': 233.5330047607422, 'audio_id': 'YOU1000000134', 'title': 'YOU1000000134', 'url': 'N/A', 'source': 2, 'category': 10, 'original_full_path': 'audio/youtube/P0000/YOU1000000134.opus'}


In [52]:
sample = test_data[0]

print(sample["text"])
print(sample["speaker"])
print(sample["begin_time"], sample["end_time"])

ONE OF THEIR STANFORD PROFESSORS USED TO SAY <COMMA> WELL <COMMA> THE DIFFERENCE BETWEEN THE TWO OF THEM WAS THAT SERGEI WOULD JUST BURST INTO MY OFFICE WITHOUT ASKING <PERIOD> LARRY WOULD KNOCK AND THEN BURST IN <PERIOD>
N/A
223.66200256347656 233.5330047607422


In [53]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = whisper.load_model("medium").to(device)

In [54]:
sample = test_data[0]
audio = sample["audio"]["array"]
result = model.transcribe(audio)
print(result["text"])

 One of their Stanford professors used to say, well, the difference between the two of them was that Sergey would just burst into my office without asking, Larry would knock, and then burst in.


In [55]:
#sf.write("sample.wav", waveform, sr)

In [56]:
print(sample["text"])

ONE OF THEIR STANFORD PROFESSORS USED TO SAY <COMMA> WELL <COMMA> THE DIFFERENCE BETWEEN THE TWO OF THEM WAS THAT SERGEI WOULD JUST BURST INTO MY OFFICE WITHOUT ASKING <PERIOD> LARRY WOULD KNOCK AND THEN BURST IN <PERIOD>


#BATCH TESTING for 50 samples

In [57]:
# device = "cuda" if torch.cuda.is_available() else "cpu"

# model = whisper.load_model("medium").to(device)
# print("Using:", device)

# batch= test_data.select(range(50))
# print("Selected 50 batches of samples")

# results=[]
# print("created a list to save the results")

# for i, sample in enumerate(batch):
#     audio = sample["audio"]["array"]
#     gt_text= sample["text"]

#     #whisper
#     pred= model.transcribe(audio)["text"]

#     results.append({ "id": sample["segment_id"],
#         "ground_truth": gt_text,
#         "prediction": pred
#         })

#     if i % 10 == 0:
#         print(f"Processed {i}/{len(batch)}")


In [58]:
# from jiwer import wer

# wers = []

# for r in results:
#     error = wer(r["ground_truth"].lower(), r["prediction"].lower())
#     wers.append(error)

# print("Average WER:", sum(wers) / len(wers))

In [59]:
# with open("whisper_results_50.json", "w")as f:
#     json.dump(results, f, indent=2)

#TESTING with the test datasets

WER=NS+D+I​/N

In [ ]:
results=[]
failed=[]
device = "cuda" if torch.cuda.is_available() else "cpu"

model = whisper.load_model("medium").to(device)

print("Running on:", device)

for i, sample in enumerate(test_data):
    try:
        audio = sample["audio"]["array"]
        gt_text= sample["text"]

        #whisper
        pred_text= model.transcribe(audio)["text"]

        results.append({
            "id": sample["segment_id"],
            "audio_id": sample["audio_id"],
            "ground_truth": gt_text,
            "prediction": pred_text,
            "begin_time": sample["begin_time"],
            "end_time": sample["end_time"]
        })

    except Exception as e:
        failed.append({
            "index": i,
            "segment_id": sample.get("segment_id", "unknown"),
            "error": str(e)
        })
        continue

    # progress logging
    if i % 100 == 0:
        print(f"Processed {i}/{len(test_data)}")

    # safety checkpoint (prevents data loss on crash)
    if i % 1000 == 0 and i > 0:
        with open("whisper_partial_results.json", "w") as f:
            json.dump(results, f, indent=2)
    

Running on: cuda
Processed 0/25619
Processed 100/25619
Processed 200/25619
Processed 300/25619
Processed 400/25619
Processed 500/25619
Processed 600/25619
Processed 700/25619
Processed 800/25619
Processed 900/25619
Processed 1000/25619
Processed 1100/25619
Processed 1200/25619
Processed 1300/25619
Processed 1400/25619
Processed 1500/25619
Processed 1600/25619
Processed 1700/25619
Processed 1800/25619
Processed 1900/25619
Processed 2000/25619
Processed 2100/25619
Processed 2200/25619
Processed 2300/25619
Processed 2400/25619
Processed 2500/25619
Processed 2600/25619
Processed 2700/25619
Processed 2800/25619
Processed 2900/25619
Processed 3000/25619
Processed 3100/25619
Processed 3200/25619
Processed 3300/25619
Processed 3400/25619
Processed 3500/25619
Processed 3600/25619
Processed 3700/25619
Processed 3800/25619
Processed 3900/25619
Processed 4000/25619
Processed 4100/25619
Processed 4200/25619
Processed 4300/25619
Processed 4400/25619
Processed 4500/25619
Processed 4600/25619
Processe

RuntimeError: getFramesPlayedInRangeAudio, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1181, No audio frames were decoded. This is probably because start_seconds is too high(0),or because stop_seconds(nullopt) is too low.

In [ ]:
with open("whisper_full_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved full results!")

Saved full results!


In [69]:
import json
failed=[]
with open("whisper_partial_results.json", "r") as f:
    results = json.load(f)

print("Loaded:", len(results), "completed samples")

start_index = len(results)

print("Resuming from:", start_index)




Loaded: 20001 completed samples
Resuming from: 20001


In [70]:
for i in range(start_index, len(test_data)):

    sample = test_data[i]

    try:
        audio = sample["audio"]["array"]
        gt_text = sample["text"]

        pred = model.transcribe(audio)["text"]

        results.append({
            "id": sample["segment_id"],
            "ground_truth": gt_text,
            "prediction": pred,
            "begin_time": sample["begin_time"],
            "end_time": sample["end_time"]
        })

    except Exception as e:
        print(f"Skipping sample {i}: {e}")

        failed.append({
            "index": i,
            "segment_id": sample.get("segment_id", "unknown"),
            "error": str(e)
        })

        continue

    # Progress logging
    if i % 100 == 0:
        print(f"Processed {i}/{len(test_data)}")

    # Save checkpoint
    if i % 500 == 0:
        with open("whisper_partial.json", "w") as f:
            json.dump(results, f, indent=2)

        with open("failed_samples.json", "w") as f:
            json.dump(failed, f, indent=2)

Skipping sample 20003: getFramesPlayedInRangeAudio, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1181, No audio frames were decoded. This is probably because start_seconds is too high(0),or because stop_seconds(nullopt) is too low.
Processed 20100/25619
Processed 20200/25619
Processed 20300/25619
Processed 20400/25619
Processed 20500/25619
Processed 20600/25619
Processed 20700/25619
Processed 20800/25619
Processed 20900/25619
Processed 21000/25619
Processed 21100/25619
Processed 21200/25619
Processed 21300/25619
Processed 21400/25619
Processed 21500/25619
Processed 21600/25619
Processed 21700/25619
Processed 21800/25619
Processed 21900/25619
Processed 22000/25619
Processed 22100/25619
Processed 22200/25619
Processed 22300/25619
Processed 22400/25619
Processed 22500/25619
Processed 22600/25619
Processed 22700/25619
Processed 22800/25619
Processed 22900/25619
Processed 23000/25619
Processed 23100/25619
Processed 23200/25619
Processed 233

In [71]:
with open("whisper_full_results.json", "w") as f:
    json.dump(results, f, indent=2)

with open("failed_samples.json", "w") as f:
    json.dump(failed, f, indent=2)

print("Finished processing all samples")

Finished processing all samples
